In [1]:
import serial
import threading
import csv
import time
from datetime import datetime

In [2]:
LLD1_PORT = "COM9"
LLD2_PORT = "COM6"

BAUD = 115200

experiment_name = input("Enter experiment name: ")

# Global control flag
running = True

Enter experiment name:  VB-1E-8_slug6


In [3]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

lld1_filename = f"LLD1_{experiment_name}_{timestamp}.csv"
lld2_filename = f"LLD2_{experiment_name}_{timestamp}.csv"

print(lld1_filename)
print(lld2_filename)

LLD1_VB-1E-8_slug6_20260513_134123.csv
LLD2_VB-1E-8_slug6_20260513_134123.csv


In [4]:
def log_serial_data(port, filename, detector_name):

    global running

    ser = serial.Serial(port, BAUD, timeout=1)

    # Allow Arduino reset
    time.sleep(2)

    print(f"{detector_name} connected on {port}")

    with open(filename, "w", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "pc_timestamp",
            "arduino_ms",
            "analog_value"
        ])

        while running:

            line = ser.readline().decode("utf-8").strip()

            if not line:
                continue

            try:

                arduino_ms, analog_value = line.split(",")

                pc_timestamp = time.time()

                writer.writerow([
                    pc_timestamp,
                    arduino_ms,
                    analog_value
                ])

            except ValueError:

                print(f"{detector_name} bad line:", line)

    ser.close()

    print(f"{detector_name} logging stopped cleanly.")

In [5]:
lld1_thread = threading.Thread(
    target=log_serial_data,
    args=(LLD1_PORT, lld1_filename, "LLD1"),
    daemon=True
)

lld2_thread = threading.Thread(
    target=log_serial_data,
    args=(LLD2_PORT, lld2_filename, "LLD2"),
    daemon=True
)

lld1_thread.start()
lld2_thread.start()

print("Logging started.")
print("Interrupt kernel to stop acquisition.")

Logging started.
Interrupt kernel to stop acquisition.
LLD1 connected on COM9
LLD2 connected on COM6
LLD2 logging stopped cleanly.
LLD1 logging stopped cleanly.


In [6]:
try:

    while True:
        time.sleep(1)

except KeyboardInterrupt:

    print("Stopping acquisition...")

    running = False

    time.sleep(2)

    print("Acquisition stopped.")

Stopping acquisition...
Acquisition stopped.
